# Dueling Networks DQN

In [1]:
import gymnasium as gym
import icu_sepsis

import numpy as np
import matplotlib
matplotlib.rcParams['agg.path.chunksize'] = 10000
import matplotlib.pyplot as plt
import os
import re
import glob
import random
from itertools import product

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

from gymnasium.vector import SyncVectorEnv

env = gym.make('Sepsis/ICU-Sepsis-v2')

state, info = env.reset()
print('Initial state:', state)
print('Extra info:', info)

next_state, reward, terminated, truncated, info = env.step(0)
print('\nTaking action 0:')
print('Next state:', next_state)
print('Reward:', reward)
print('Terminated:', terminated)
print('Truncated:', truncated)

print("Actions and State Space:")

print("Action space:", env.action_space)
print("State space:", env.observation_space)

Initial state: 624
Extra info: {'admissible_actions': [0], 'state_vector': array([-0.14705882, -0.24509804, -0.48598039,  0.55098039,  0.33476986,
       -0.05058546, -0.53187757, -0.29714864, -0.34412376, -0.41080984,
       -0.40632826, -0.5763351 , -0.05612998,  2.16204785,  1.639067  ,
       -0.10847038,  0.12239778, -0.03820453,  0.77307535,  0.42407934,
       -0.21947964, -0.31108738, -0.07084028,  0.20786207, -0.11107361,
        0.02029516,  2.90174416, -0.07583882, -0.01038845, -0.35582086,
       -0.0508786 ,  0.54030911, -0.69136494, -0.03703611,  0.45811431,
       -0.11752978,  0.39799485,  0.3939501 ,  0.48958091,  0.06275602,
        0.04731095,  0.13756509, -0.07728013, -3.00335102, -3.00886387,
       -2.41291891, -1.81679327]), 'sofa_score': np.float64(7.898395721925134)}

Taking action 0:
Next state: 648
Reward: 0.0
Terminated: False
Truncated: False
Actions and State Space:
Action space: Discrete(25)
State space: Discrete(716)


In [11]:
# Prioritized Replay Buffer to store transitions
class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.pos = 0
        self.buffer = []
        self.priorities = []

    def add(self, s, a, r, s2, done):
        max_prio = max(self.priorities, default=1.0)
        if len(self.buffer) < self.capacity:
            self.buffer.append((s, a, r, s2, done))
            self.priorities.append(max_prio)
        else:
            self.buffer[self.pos] = (s, a, r, s2, done)
            self.priorities[self.pos] = max_prio
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size, beta=0.4):
        prios = np.array(self.priorities) ** self.alpha
        probs = prios / prios.sum()
        idxs = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[i] for i in idxs]
        total = len(self.buffer)
        weights = (total * probs[idxs]) ** (-beta)
        weights /= weights.max()
        return samples, idxs, weights

    def update_priorities(self, idxs, errors, eps=1e-6):
        for i, e in zip(idxs, errors):
            self.priorities[i] = abs(e) + eps

def init_weights_he(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)

# Dueling DQN Network
class DuelingQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(DuelingQNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc_val = nn.Linear(hidden_dim, hidden_dim)
        self.fc_adv = nn.Linear(hidden_dim, hidden_dim)
        self.val_out = nn.Linear(hidden_dim, 1)
        self.adv_out = nn.Linear(hidden_dim, action_dim)
        self.apply(init_weights_he)

    def forward(self, x):   
        x  = F.relu(self.fc1(x))
        v  = F.relu(self.fc_val(x))
        a  = F.relu(self.fc_adv(x))
        v  = self.val_out(v)                  # (B,1)
        a  = self.adv_out(a)                  # (B,|A|)
        q  = v + a - a.mean(dim=1, keepdim=True)
        return q
    
# Dueling DQN Agent
class DuelingDQNAgent():

    def __init__(self,
                 env_name = 'Sepsis/ICU-Sepsis-v2',
                 lr = 1e-3,
                 gamma = 0.99,
                 buffer_size = 20_000,
                 batch_size = 64,
                 eps_start = 1.0,
                 eps_end = 0.01,
                 eps_decay_steps = 50_000,
                 alpha = 0.6,   beta=0.4,
                 learning_starts = 1_000,
                 train_freq = 10,
                 target_update = 1_000,
                 device = 'cpu'):
        
        # environment
        self.env = gym.make(env_name)
        self.state_dim = self.env.observation_space.n
        self.action_dim = self.env.action_space.n
        self.device = torch.device(device)
        self.gamma = gamma

        # deuling networks
        self.q_net      = DuelingQNetwork(self.state_dim,self.action_dim).to(self.device)
        self.target_net = DuelingQNetwork(self.state_dim,self.action_dim).to(self.device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.opt = optim.Adam(self.q_net.parameters(), lr=lr)

        # replay buffer
        self.buffer = PrioritizedReplayBuffer(buffer_size, alpha)
        self.batch_size = batch_size

        # misc
        self.eps_s, self.eps_e = eps_start, eps_end
        self.eps_decay_steps   = eps_decay_steps
        self.beta = beta
        self.learning_starts = learning_starts
        self.train_freq = train_freq
        self.target_update = target_update

        self.total_steps = 0
        self.learn_step = 0

    # ε-greedy
    def select_action(self, state):
        ε = self.epsilon()
        if random.random() < ε:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            s = F.one_hot(torch.tensor(state), self.state_dim).float().to(self.device)
            q = self.q_net(s.unsqueeze(0))
            return q.argmax(dim=1).item()

    # epsilon decay schedule
    def epsilon(self):
        return max(self.eps_e,
                   self.eps_s - (self.eps_s-self.eps_e) *
                   min(1.0, self.total_steps/self.eps_decay_steps))
    
    # one complete episode
    def train_one_episode(self):
        state,_ = self.env.reset()
        done=False; ret=0; length=0; inad=0
        while not done:
            length+=1
            action = self.select_action(state)
            next_state, reward, term, trunc, info = self.env.step(action)
            done = term or trunc
            ret  += reward
            inad += int(info.get('inadmissible',False))
            self.buffer.add(state,action,reward,next_state,done)
            state = next_state
            self.total_steps += 1
            # update
            if (self.total_steps>=self.learning_starts and
                self.total_steps % self.train_freq == 0 and
                len(self.buffer.buffer)>=self.batch_size):
                self.learn() # encapsulated in learn()
        return ret, length, inad/length

    def learn(self):
        samples, idxs, w = self.buffer.sample(self.batch_size, self.beta)
        s,a,r,ns,d = map(np.array, zip(*samples))
        sb  = F.one_hot(torch.tensor(s),  self.state_dim).float().to(self.device)
        nsb = F.one_hot(torch.tensor(ns), self.state_dim).float().to(self.device)
        ab  = torch.tensor(a).unsqueeze(1).to(self.device)
        rb  = torch.tensor(r, dtype=torch.float32).unsqueeze(1).to(self.device)
        db  = torch.tensor(d, dtype=torch.float32).unsqueeze(1).to(self.device)
        wb  = torch.tensor(w).unsqueeze(1).to(self.device)
        # Q(s,a)
        q  = self.q_net(sb).gather(1,ab)
        # target
        with torch.no_grad():
            maxi = self.target_net(nsb).max(1,keepdim=True)[0]
            tgt  = rb + self.gamma*maxi*(1.0-db)
        td_err = (tgt - q).squeeze(1).detach().cpu().numpy()
        loss = (wb * (q-tgt).pow(2)).mean()
        # optimise
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        self.buffer.update_priorities(idxs, td_err)
        # target net
        self.learn_step += 1
        if self.learn_step % self.target_update == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

In [12]:
# reproducibility
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

env_name = 'Sepsis/ICU-Sepsis-v2'
agent = DuelingDQNAgent(
    env_name = 'Sepsis/ICU-Sepsis-v2',
    lr = 1e-3,
    gamma = 0.99,
    buffer_size = 10_000,
    batch_size = 64,
    eps_start = 1.0,
    eps_end = 0.001,
    eps_decay_steps = 50_000,
    alpha = 0.6,   beta=0.4,
    learning_starts = 10_000,
    train_freq = 10,
    target_update = 5_000,
    device = 'cpu')

N_EPISODES = 300_000
returns, lengths, inad_freqs = [], [], []

pbar = tqdm(range(1, N_EPISODES+1), desc='Dueling DQN Train', unit='ep')
for ep in pbar:
    r, l, inad = agent.train_one_episode()
    returns.append(r)
    lengths.append(l)
    inad_freqs.append(inad)

    pbar.set_postfix({"r":f"{r:.2f}", "len":l, "inad%":f"{inad:.2%}"})

Dueling DQN Train:   2%|▏         | 5062/300000 [05:47<5:37:29, 14.57ep/s, r=1.00, len=3, inad%=0.00%]  


KeyboardInterrupt: 